<a href="https://colab.research.google.com/github/boluwatifeakintayo/boluwatife-FLYRANKAI-repo/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/boluwatifeakintayo/boluwatife-FLYRANKAI-repo/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
hf_token = userdata.get('flyrank-w3')

import duckdb
con = duckdb.connect()
con.sql("INSTALL httpfs")
con.sql("LOAD httpfs")
con.sql(f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
""")
print("Setup complete")

Setup complete


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

**Plain-words rule:** A page is worth reviewing if it hasn't been updated in
a long time (180+ days) but is still getting real search impressions —
meaning there's an audience worth protecting, not a dead page.

**Reason codes it can output:**
- `stale_but_visible` — old content, still getting real impressions (highest priority)
- `stale_and_dead` — old content, very low impressions (lower priority, may not be worth fixing)
- `not_stale` — recently updated, not flagged by this rule

In [3]:
march_agg = con.sql("""
    SELECT content_hash_id, SUM(gsc_clicks) AS clicks_march, SUM(gsc_impressions) AS impressions_march
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

april_agg = con.sql("""
    SELECT content_hash_id, SUM(gsc_clicks) AS clicks_april
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

merged = march_agg.merge(april_agg, on="content_hash_id", how="inner")
merged["is_declining"] = (merged["clicks_april"] < merged["clicks_march"]).astype(int)
print(f"{len(merged)} pages labeled")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

158549 pages labeled


In [4]:
# Get content update dates
content_dates = con.sql("""
    SELECT content_hash_id, content_updated_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
""").df()

# Join onto our labeled pages
staleness_check = merged.merge(content_dates, on="content_hash_id", how="inner")

# "Days since update" as of end of March (our decision moment)
import pandas as pd
reference_date = pd.Timestamp("2026-03-31")
staleness_check["days_since_update"] = (reference_date - pd.to_datetime(staleness_check["content_updated_date"])).dt.days

# Bucket: stale (180+ days) vs fresh
staleness_check["stale"] = (staleness_check["days_since_update"] >= 180).astype(int)

# THE ACTUAL CHECK: compare decline rate between buckets
bucket_table = staleness_check.groupby("stale")["is_declining"].agg(
    decline_rate="mean",
    n="count"
)
print(bucket_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       decline_rate       n
stale                      
0          0.279652  158397
1          0.059211     152


**Signal 1 — Staleness**

Claim: Pages not updated in 180+ days are more likely to be declining.

Bucket table: fresh (n=158,397) → 27.97% decline rate. stale (n=152) → 5.92%
decline rate.

Verdict: OPPOSITE — but with a critical caveat. Only 152 of 158,549 pages
(0.1%) qualify as "stale" by this threshold, meaning content_updated_date is
clustered very recently across nearly the entire dataset. This sample size is
too small to trust the direction of the effect. The real finding here isn't
"staleness prevents decline" — it's that this dataset doesn't have enough
genuinely stale content to test the staleness hypothesis meaningfully at all.

In [7]:
# Get March position and CTR data per page
position_data = con.sql("""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position_march,
        SUM(gsc_clicks) AS clicks_march_ctr,
        SUM(gsc_impressions) AS impressions_march_ctr
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

position_data["ctr_march"] = position_data["clicks_march_ctr"] / position_data["impressions_march_ctr"]

# Bucket pages into position tiers, same style as your Week 1 discovery
import numpy as np
position_data["position_tier"] = pd.cut(
    position_data["avg_position_march"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"]
)

# Expected CTR = the AVERAGE ctr for each position tier
expected_ctr = position_data.groupby("position_tier")["ctr_march"].transform("mean")
position_data["ctr_below_expected"] = (position_data["ctr_march"] < expected_ctr).astype(int)

print(position_data[["avg_position_march", "ctr_march", "position_tier", "ctr_below_expected"]].head(10))
expected_ctr = position_data.groupby("position_tier", observed=True)["ctr_march"].transform("mean")

   avg_position_march  ctr_march position_tier  ctr_below_expected
0            5.145765   0.001112        page_1                   1
1            4.909314   0.000000        page_1                   1
2            5.177774   0.000000        page_1                   1
3            4.685335   0.001295        page_1                   1
4            4.266667   0.000000        page_1                   1
5           17.148172   0.002049        page_2                   1
6           10.479603   0.000000        page_2                   1
7            7.926052   0.000000        page_1                   1
8           10.127319   0.000000        page_2                   1
9            5.467633   0.000000        page_1                   1


/tmp/ipykernel_2762/2240755761.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  expected_ctr = position_data.groupby("position_tier")["ctr_march"].transform("mean")


In [6]:
ctr_check = merged.merge(position_data, on="content_hash_id", how="inner")

bucket_table_ctr = ctr_check.groupby("ctr_below_expected")["is_declining"].agg(
    decline_rate="mean",
    n="count"
)
print(bucket_table_ctr)

                    decline_rate       n
ctr_below_expected                      
0                       0.681156   25605
1                       0.202070  132944


**Signal 2 — CTR-vs-Position**

Claim: Pages with CTR below what's expected for their ranking position are
more likely to be declining.

Bucket table: below-expected CTR (n=132,944) → 20.2% decline rate.
at/above-expected CTR (n=25,605) → 68.1% decline rate.

Verdict: OPPOSITE — and unlike the staleness check, this has strong sample
sizes on both sides, so it's not a small-n fluke. The direction is genuinely
surprising and worth flagging as needing deeper investigation rather than
trusted at face value; one possibility is that "expected CTR" itself is being
pulled toward zero by a large share of near-zero-CTR pages, making the
"below expected" bucket behave differently than intended. I'm treating this
as a real, provable negative result rather than building my rule on this
signal.

## 1. My rule and its reason codes (final, after signal checks)

**Plain-words rule:** Flag a page for review if its click-through rate is at
or above what's typically expected for its ranking position — this group
showed a 68.1% decline rate vs. 20.2% for below-expected pages (n=25,605 vs
n=132,944), a strong, well-powered, if counterintuitive, signal.

**Note on staleness:** Originally considered as a second signal, but
disqualified — only 152 of 158,549 pages qualified as "stale" (180+ days),
too small a sample to trust.

**Reason codes it can output:**
- `ctr_at_or_above_expected` — flagged; this group has the high decline rate
- `ctr_below_expected` — not flagged; this group is comparatively stable

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# Start from the data we already built during the CTR signal check
ctr_check = merged.merge(position_data, on="content_hash_id", how="inner")

# THE RULE, as a transparent score — simple, readable, no fitted weights
# Higher score = more urgent to review
# We use impressions as a multiplier so pages with more visibility (more "worth protecting") rank higher
ctr_check["score"] = ctr_check["ctr_below_expected"].apply(lambda x: 0 if x == 1 else 1) * ctr_check["impressions_march_ctr"]

# Reason codes, matching what we decided in Section 1
ctr_check["reason_code"] = ctr_check["ctr_below_expected"].apply(
    lambda x: "ctr_below_expected" if x == 1 else "ctr_at_or_above_expected"
)

# Action label — simple, single action for now
ctr_check["action"] = ctr_check["reason_code"].apply(
    lambda x: "review_content" if x == "ctr_at_or_above_expected" else "monitor_only"
)

# Rank everything by score, highest first
ranked = ctr_check.sort_values("score", ascending=False)

print(ranked[["content_hash_id", "score", "reason_code", "action", "is_declining"]].head(10))

                 content_hash_id     score               reason_code  \
5891    content_e7b5dd4dff461ad2  205045.0  ctr_at_or_above_expected   
35985   content_f107e54b10b43725  195997.0  ctr_at_or_above_expected   
113163  content_512dbad65bd5ade9  154358.0  ctr_at_or_above_expected   
56228   content_66288edeb93b7c4f  137878.0  ctr_at_or_above_expected   
136607  content_0ec90963d98b97a5  130338.0  ctr_at_or_above_expected   
119148  content_57768353f230d65d  114479.0  ctr_at_or_above_expected   
57755   content_5fa2737c68998c2e  109043.0  ctr_at_or_above_expected   
32998   content_b2cb08ff59fcce78  106025.0  ctr_at_or_above_expected   
24528   content_f86f77b3ebdc05ee  105420.0  ctr_at_or_above_expected   
29697   content_ef7013c86d07aa99  104330.0  ctr_at_or_above_expected   

                action  is_declining  
5891    review_content             1  
35985   review_content             1  
113163  review_content             1  
56228   review_content             1  
136607  revi

In [9]:
import os

# Make sure the folder exists first (fresh Colab sessions won't have it yet)
os.makedirs("work/outputs", exist_ok=True)

# Save the full ranked queue
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Saved. Row count:", len(ranked))

Saved. Row count: 158549


In [10]:
top10 = ranked.head(10)[["content_hash_id", "score", "reason_code", "action", "is_declining", "avg_position_march", "ctr_march"]]
print(top10.to_string())

                 content_hash_id     score               reason_code          action  is_declining  avg_position_march  ctr_march
5891    content_e7b5dd4dff461ad2  205045.0  ctr_at_or_above_expected  review_content             1            4.544203   0.011929
35985   content_f107e54b10b43725  195997.0  ctr_at_or_above_expected  review_content             1            3.186054   0.005082
113163  content_512dbad65bd5ade9  154358.0  ctr_at_or_above_expected  review_content             1            3.019798   0.016235
56228   content_66288edeb93b7c4f  137878.0  ctr_at_or_above_expected  review_content             1           18.615742   0.005672
136607  content_0ec90963d98b97a5  130338.0  ctr_at_or_above_expected  review_content             1            3.249691   0.011708
119148  content_57768353f230d65d  114479.0  ctr_at_or_above_expected  review_content             1            3.326894   0.005075
57755   content_5fa2737c68998c2e  109043.0  ctr_at_or_above_expected  review_content      

## 3. Top-10 review

1. **content_e7b5dd4dff461ad2** (score 205,045, position 4.5, CTR 1.19%) —
   Action: review_content. Why: CTR at/above expected for its position, high
   impressions drove the top score. Declining: yes. What would make it wrong:
   if this page's low CTR is due to a misleading title/snippet rather than an
   early decline signal — a metadata fix might be the real solution, not a
   content rewrite.

2. **content_f107e54b10b43725** (score 195,997, position 3.2, CTR 0.51%) —
   Action: review_content. Why: same rule logic, very high impressions.
   Declining: yes. What would make it wrong: an extremely low CTR (0.51%) at
   position 3 is unusual enough that it might indicate a tracking/data issue
   rather than genuine underperformance — worth spot-checking before trusting.

3. **content_512dbad65bd5ade9** (score 154,358, position 3.0, CTR 1.62%) —
   Action: review_content. Why: same logic. Declining: yes. What would make
   it wrong: if this page ranks #3 for a branded/navigational query where
   low CTR is normal (searchers already know where they're going), flagging
   it would be a false alarm.

4. **content_66288edeb93b7c4f** (score 137,878, position 18.6, CTR 0.57%) —
   Action: review_content. Why: same logic, though this page's position
   (18.6) is notably worse than the others in the top 10. Declining: yes.
   What would make it wrong: nothing here — this one actually looks like a
   legitimately weak page (bad position AND bad CTR), a reasonably solid pick.

5. **content_0ec90963d98b97a5** (score 130,338, position 3.2, CTR 1.17%) —
   Action: review_content. Why: same logic. Declining: yes. What would make
   it wrong: same concern as row 3 — could be a branded query with naturally
   low CTR despite good position.

6. **content_57768353f230d65d** (score 114,479, position 3.3, CTR 0.51%) —
   Action: review_content. Why: same logic. Declining: yes. What would make
   it wrong: same tracking-accuracy concern as row 2 — CTR this low at
   position 3 is worth a manual spot-check.

7. **content_5fa2737c68998c2e** (score 109,043, position 3.8, CTR 0.97%) —
   Action: review_content. Why: same logic. **Declining: NO.** What would
   make it wrong: it already is wrong — this page was flagged as high-priority
   but did not actually decline. A clean example of a false positive.

8. **content_b2cb08ff59fcce78** (score 106,025, position 3.2, CTR 0.63%) —
   Action: review_content. Why: same logic. Declining: yes. What would make
   it wrong: same concern as rows 2/6 — unusually low CTR at a strong
   position warrants checking for a tracking or snippet issue first.

9. **content_f86f77b3ebdc05ee** (score 105,420, position 3.9, CTR 0.52%) —
   Action: review_content. Why: same logic. Declining: yes. What would make
   it wrong: same as above — very low CTR for a strong position, worth
   verifying it's a real content issue and not a data artifact.

10. **content_ef7013c86d07aa99** (score 104,330, position 4.7, CTR 0.92%) —
    Action: review_content. Why: same logic. **Declining: NO.** What would
    make it wrong: it already is wrong — a second confirmed false positive
    in the top 10.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
top20 = ranked.head(20)[["content_hash_id", "score", "reason_code", "action", "is_declining", "avg_position_march", "ctr_march"]]
print(top20.to_string())

                 content_hash_id     score               reason_code          action  is_declining  avg_position_march  ctr_march
5891    content_e7b5dd4dff461ad2  205045.0  ctr_at_or_above_expected  review_content             1            4.544203   0.011929
35985   content_f107e54b10b43725  195997.0  ctr_at_or_above_expected  review_content             1            3.186054   0.005082
113163  content_512dbad65bd5ade9  154358.0  ctr_at_or_above_expected  review_content             1            3.019798   0.016235
56228   content_66288edeb93b7c4f  137878.0  ctr_at_or_above_expected  review_content             1           18.615742   0.005672
136607  content_0ec90963d98b97a5  130338.0  ctr_at_or_above_expected  review_content             1            3.249691   0.011708
119148  content_57768353f230d65d  114479.0  ctr_at_or_above_expected  review_content             1            3.326894   0.005075
57755   content_5fa2737c68998c2e  109043.0  ctr_at_or_above_expected  review_content      

## 3. Top-20 review

1. **content_e7b5dd4dff461ad2** — score 205,045, pos 4.5, CTR 1.19%. Action:
   review_content. Reason: ctr_at_or_above_expected. Confidence: Medium —
   high impressions support the pick, but CTR this low for position 4-5 is
   unusual. Declining: yes. Wrong if: low CTR is a metadata/title issue, not
   a decline signal.

2. **content_f107e54b10b43725** — score 195,997, pos 3.2, CTR 0.51%. Action:
   review_content. Reason: ctr_at_or_above_expected. Confidence: Low — 0.51%
   CTR at position 3 is extreme; possible data/tracking issue. Declining:
   yes. Wrong if: CTR figure is unreliable rather than a real signal.

3. **content_512dbad65bd5ade9** — score 154,358, pos 3.0, CTR 1.62%. Action:
   review_content. Reason: ctr_at_or_above_expected. Confidence: Medium.
   Declining: yes. Wrong if: this is a branded/navigational query where low
   CTR is normal and expected.

4. **content_66288edeb93b7c4f** — score 137,878, pos 18.6, CTR 0.57%. Action:
   review_content. Reason: ctr_at_or_above_expected. Confidence: High — weak
   position AND weak CTR together is a coherent, believable decline story.
   Declining: yes. Wrong if: nothing obvious — this is a solid pick.

5. **content_0ec90963d98b97a5** — score 130,338, pos 3.2, CTR 1.17%. Action:
   review_content. Reason: ctr_at_or_above_expected. Confidence: Medium.
   Declining: yes. Wrong if: branded query effect, same as row 3.

6. **content_57768353f230d65d** — score 114,479, pos 3.3, CTR 0.51%. Action:
   review_content. Reason: ctr_at_or_above_expected. Confidence: Low — same
   extreme-low-CTR concern as row 2. Declining: yes. Wrong if: tracking
   issue rather than real underperformance.

7. **content_5fa2737c68998c2e** — score 109,043, pos 3.8, CTR 0.97%. Action:
   review_content. Reason: ctr_at_or_above_expected. Confidence: Low —
   **already wrong**. Declining: NO. This is a confirmed false positive.

8. **content_b2cb08ff59fcce78** — score 106,025, pos 3.2, CTR 0.63%. Action:
   review_content. Reason: ctr_at_or_above_expected. Confidence: Low, same
   extreme-CTR pattern. Declining: yes. Wrong if: data/tracking issue.

9. **content_f86f77b3ebdc05ee** — score 105,420, pos 3.9, CTR 0.52%. Action:
   review_content. Reason: ctr_at_or_above_expected. Confidence: Low, same
   pattern as row 2/6/8. Declining: yes. Wrong if: tracking issue.

10. **content_ef7013c86d07aa99** — score 104,330, pos 4.7, CTR 0.92%. Action:
    review_content. Reason: ctr_at_or_above_expected. Confidence: Low —
    **already wrong**.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

**Weak picks identified:**

1. **Rows 7, 10, 18** (content_5fa2737c68998c2e, content_ef7013c86d07aa99,
   content_6c0351d021143c7a) — confirmed false positives. All three were
   flagged with high scores but did NOT actually decline. These represent
   real failures of the rule, not edge cases — 3 out of the top 20 (15%)
   were wrong.

2. **The extreme low-CTR pattern across most of the top 20** — 14 of 20 rows
   have CTR under 1%, despite ranking in positions 3-5 (where Week 1 data
   showed expected CTR closer to 30-35%). This is suspicious enough that I
   don't fully trust these picks even where `is_declining = 1` matched — the
   rule may be latching onto a measurement quirk (possibly how CTR is
   calculated or reported for certain page/query types) rather than a
   genuine content-quality signal. This weakens confidence in most of the
   top 20, even the "correct" predictions.

In [12]:
print("Columns used to build the score:")
print(["avg_position_march", "ctr_march", "impressions_march_ctr", "ctr_below_expected"])
print("\nAll of these are aggregated ONLY from March 2026 data — none touch April.")

# Quick proof: confirm the score column has zero correlation-by-construction with clicks_april directly
# (it shouldn't, since April was never referenced when building score)
print("\nColumns present in ranked dataframe:")
print(ranked.columns.tolist())

Columns used to build the score:
['avg_position_march', 'ctr_march', 'impressions_march_ctr', 'ctr_below_expected']

All of these are aggregated ONLY from March 2026 data — none touch April.

Columns present in ranked dataframe:
['content_hash_id', 'clicks_march', 'impressions_march', 'clicks_april', 'is_declining', 'avg_position_march', 'clicks_march_ctr', 'impressions_march_ctr', 'ctr_march', 'position_tier', 'ctr_below_expected', 'score', 'reason_code', 'action']


**Leakage check:** `clicks_april` is present in the working dataframe (since
it was needed to construct the `is_declining` label), but it was never
referenced in the score formula itself. The score is calculated purely from
`ctr_below_expected` and `impressions_march_ctr`, both aggregated only from
March 2026. Verified directly from the score-building code:
`score = ctr_below_expected_flag * impressions_march_ctr` — no April column
appears on the right-hand side of that line. No product flags were used
either, since none exist in this table. This is a genuine, checked
distinction, not just an assumption — `clicks_april`'s presence in the
dataframe is expected (it's needed for the label) and does not by itself
indicate leakage, since it was excluded from the actual scoring calculation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.